# 03 - Feature Engineering

Part of the **Data Science Learning** project. Explanations, comments, and notebook messages are in English.

## Learning objectives
- Create useful features from existing columns.
- Encode categorical data.
- Scale numeric features without leaking test data into training.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_diabetes, make_blobs
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42

## 2. Create an example dataset

In [2]:
df = pd.DataFrame({
    "age": [19, 25, 31, 42, 36, 28, 51, 47, 23, 39],
    "monthly_spend": [35, 60, 80, 120, 95, 55, 150, 135, 45, 105],
    "visits_per_month": [2, 4, 5, 8, 6, 3, 10, 9, 3, 7],
    "membership": ["Basic", "Basic", "Plus", "Premium", "Plus", "Basic", "Premium", "Premium", "Basic", "Plus"],
    "renewed": [0, 0, 1, 1, 1, 0, 1, 1, 0, 1]
})
df

,age,monthly_spend,visits_per_month,membership,renewed
0,19,35,2,Basic,0
1,25,60,4,Basic,0
2,31,80,5,Plus,1
3,42,120,8,Premium,1
4,36,95,6,Plus,1
5,28,55,3,Basic,0
6,51,150,10,Premium,1
7,47,135,9,Premium,1
8,23,45,3,Basic,0
9,39,105,7,Plus,1


## 3. Create and encode features

In [3]:
df["spend_per_visit"] = df["monthly_spend"] / df["visits_per_month"]
df["age_group"] = pd.cut(df["age"], bins=[0, 25, 40, 100], labels=["young", "adult", "older"])
encoded = pd.get_dummies(df, columns=["membership", "age_group"], dtype=int)
encoded

,age,monthly_spend,visits_per_month,renewed,spend_per_visit,membership_Basic,membership_Plus,membership_Premium,age_group_young,age_group_adult,age_group_older
0,19,35,2,0,17.500000,1,0,0,1,0,0
1,25,60,4,0,15.000000,1,0,0,1,0,0
2,31,80,5,1,16.000000,0,1,0,0,1,0
3,42,120,8,1,15.000000,0,0,1,0,0,1
4,36,95,6,1,15.833333,0,1,0,0,1,0
5,28,55,3,0,18.333333,1,0,0,0,1,0
6,51,150,10,1,15.000000,0,0,1,0,0,1
7,47,135,9,1,15.000000,0,0,1,0,0,1
8,23,45,3,0,15.000000,1,0,0,1,0,0
9,39,105,7,1,15.000000,0,1,0,0,1,0


## 4. Split before fitting transformations

In [4]:
X = df.drop(columns="renewed")
y = df["renewed"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)

X_train = pd.get_dummies(X_train, columns=["membership", "age_group"], dtype=int)
X_test = pd.get_dummies(X_test, columns=["membership", "age_group"], dtype=int)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

numeric_cols = ["age", "monthly_spend", "visits_per_month", "spend_per_visit"]
scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])
X_train.head()

,age,monthly_spend,visits_per_month,spend_per_visit,membership_Basic,membership_Plus,membership_Premium,age_group_young,age_group_adult,age_group_older
3,0.852651,0.899423,0.911296,-0.564076,0,0,1,0,0,1
8,-0.919265,-1.027912,-0.964901,-0.564076,1,0,0,1,0,0
1,-0.732747,-0.642445,-0.589662,-0.564076,1,0,0,1,0,0
6,1.691980,1.670358,1.661774,-0.564076,0,0,1,0,0,1
9,0.572875,0.513956,0.536056,-0.564076,0,1,0,0,1,0


## Key takeaways
- Feature engineering can expose useful patterns.
- Encoding turns categories into numeric columns.
- Fit learned transformations on training data only.

## Exercises
1. Add annual spending as a feature.
2. Try different age-group boundaries.
3. Compare a model with and without `spend_per_visit`.